# 3-Head Interview Question Classifier

DistilBERT с тремя головами:
- **Head 2**: Actionable vs non-actionable (бинарная) — основная задача
- **Head 3**: Query span detection (token-level) — где начинается вопрос

Оптимизации: pre-tokenization, mixed precision (AMP), pin_memory, persistent_workers, vectorized head3 labels.

In [ ]:
!pip install -q transformers scikit-learn tqdm

## 1. Загрузка датасета

In [ ]:
import json, os, urllib.request

URL  = ("https://raw.githubusercontent.com/russianoracle/"
        "interview-question-classifier/main/dataset/processed/dataset_normalized.jsonl")
PATH = "dataset_normalized.jsonl"

if not os.path.exists(PATH):
    print("Скачиваю датасет...")
    urllib.request.urlretrieve(URL, PATH)

with open(PATH) as f:
    samples = [json.loads(l) for l in f if l.strip()]

act  = sum(1 for s in samples if s['head2_label'] == 'actionable')
print(f"Загружено: {len(samples)}  (actionable={act}, non_actionable={len(samples)-act})")

## 2. Импорты

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()  # mixed precision только на GPU
MODEL_NAME = "distilbert-base-multilingual-cased"

print(f"PyTorch {torch.__version__} | device={DEVICE} | AMP={USE_AMP}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Pre-tokenization (один раз, не в __getitem__)

In [ ]:
MAX_LEN   = 128
LABEL2ID  = {"non_actionable": 0, "actionable": 1}
ID2LABEL  = {0: "non_actionable", 1: "actionable"}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Батчевая токенизация всего корпуса сразу ─────────────────────────────────
print("Токенизирую весь корпус...")
texts = [s['text'] for s in samples]
enc_all = tokenizer(
    texts,
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
    return_offsets_mapping=False,
)
# enc_all: {input_ids: (N, L), attention_mask: (N, L)}

# ── Head2 labels ──────────────────────────────────────────────────────────────
head2_all = torch.tensor([LABEL2ID[s['head2_label']] for s in samples], dtype=torch.long)

# ── Head3 labels (векторизовано через numpy) ──────────────────────────────────
# word_ids нужны только для построения span-маски; делаем это батчем
print("Строю Head3 span labels...")
head3_all = np.zeros((len(samples), MAX_LEN), dtype=np.int32)

for i, s in enumerate(samples):
    qs = s.get('head3_query_start') or 0
    if qs <= 0:
        continue
    word_ids = enc_all.word_ids(batch_index=i)   # list len=MAX_LEN
    wids = np.array([w if w is not None else -1 for w in word_ids], dtype=np.int32)
    head3_all[i] = (wids >= qs).astype(np.int32)

head3_all = torch.from_numpy(head3_all).long()

print(f"Готово. input_ids shape: {enc_all['input_ids'].shape}")
print(f"head2 shape: {head2_all.shape}  head3 shape: {head3_all.shape}")

## 4. Dataset (только индексирование тензоров)

In [ ]:
class QuestionDataset(Dataset):
    """Хранит предтокенизированные тензоры, __getitem__ — O(1)."""
    def __init__(self, indices):
        idx = torch.tensor(indices, dtype=torch.long)
        self.input_ids      = enc_all['input_ids'][idx]
        self.attention_mask = enc_all['attention_mask'][idx]
        self.head2          = head2_all[idx]
        self.head3          = head3_all[idx]

    def __len__(self):
        return len(self.head2)

    def __getitem__(self, i):
        return (
            self.input_ids[i],
            self.attention_mask[i],
            self.head2[i],
            self.head3[i],
        )


# Стратифицированный сплит
labels = [s['head2_label'] for s in samples]
tr_idx, tmp_idx = train_test_split(range(len(samples)), test_size=0.2, stratify=labels, random_state=42)
va_idx, te_idx  = train_test_split(tmp_idx, test_size=0.5,
                                   stratify=[labels[i] for i in tmp_idx], random_state=42)

train_ds = QuestionDataset(tr_idx)
val_ds   = QuestionDataset(va_idx)
test_ds  = QuestionDataset(te_idx)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## 5. DataLoader-ы

In [ ]:
BATCH_SIZE = 64   # можно поднять до 128 на A100

# Взвешенный сэмплер по head2
tr_labels = train_ds.head2.numpy()
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=tr_labels)
cw_tensor = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
sampler = WeightedRandomSampler(cw[tr_labels], len(tr_labels), replacement=True)

loader_kw = dict(
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)
train_loader = DataLoader(train_ds, sampler=sampler, **loader_kw)
val_loader   = DataLoader(val_ds,   shuffle=False,   **loader_kw)
test_loader  = DataLoader(test_ds,  shuffle=False,   **loader_kw)

print(f"Batches: train={len(train_loader)}  val={len(val_loader)}")
print(f"Class weights: non_act={cw[0]:.3f}  act={cw[1]:.3f}")

## 6. Модель

In [ ]:
class QuestionClassifier(nn.Module):
    def __init__(self, model_name: str, hidden: int = 256, dropout: float = 0.2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        H = self.backbone.config.hidden_size
        # Общий проекционный слой — переиспользуется обеими головами
        self.proj = nn.Sequential(nn.Linear(H, hidden), nn.GELU(), nn.Dropout(dropout))
        self.head2 = nn.Linear(hidden, 2)   # CLS → binary
        self.head3 = nn.Linear(hidden, 2)   # tokens → span

    def forward(self, input_ids, attention_mask):
        seq = self.backbone(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state  # (B,L,H)
        proj = self.proj(seq)           # (B,L,hidden)
        return self.head2(proj[:, 0]), self.head3(proj)  # (B,2), (B,L,2)


model = QuestionClassifier(MODEL_NAME).to(DEVICE)

total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Параметры: {total/1e6:.1f}M total / {trainable/1e6:.1f}M trainable")

## 7. Обучение с AMP

In [ ]:
LR         = 2e-5
NUM_EPOCHS = 4
WARMUP     = 300

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP,
    num_training_steps=len(train_loader) * NUM_EPOCHS,
)
scaler = GradScaler(enabled=USE_AMP)
loss_h2 = nn.CrossEntropyLoss(weight=cw_tensor)
loss_h3 = nn.CrossEntropyLoss()


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    # Накапливаем предсказания как тензоры (без .numpy() в цикле)
    preds_list, labels_list = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, leave=False, dynamic_ncols=True)
        for ids, mask, y2, y3 in pbar:
            ids, mask = ids.to(DEVICE, non_blocking=True), mask.to(DEVICE, non_blocking=True)
            y2, y3    = y2.to(DEVICE, non_blocking=True),  y3.to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):
                logits2, logits3 = model(ids, mask)
                loss = 0.7 * loss_h2(logits2, y2) + \
                       0.3 * loss_h3(logits3.flatten(0, 1), y3.flatten())

            if train:
                optimizer.zero_grad(set_to_none=True)  # быстрее чем zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            total_loss += loss.item()
            preds_list.append(logits2.argmax(1))
            labels_list.append(y2)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    # Конкатенация тензоров (быстрее, чем extend списка numpy)
    all_preds  = torch.cat(preds_list).cpu().numpy()
    all_labels = torch.cat(labels_list).cpu().numpy()
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return total_loss / len(loader), acc, f1


best_f1, patience, pat_cnt = 0.0, 2, 0

for epoch in range(NUM_EPOCHS):
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader,   train=False)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  "
          f"train: loss={tr_loss:.4f} acc={tr_acc:.4f}  "
          f"val: loss={va_loss:.4f} acc={va_acc:.4f} f1={va_f1:.4f}")

    if va_f1 > best_f1:
        best_f1, pat_cnt = va_f1, 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✅ checkpoint сохранён (F1={va_f1:.4f})")
    else:
        pat_cnt += 1
        if pat_cnt >= patience:
            print("  Early stop.")
            break

model.load_state_dict(torch.load("best_model.pth"))
print("Обучение завершено.")

## 8. Тест

In [ ]:
te_loss, te_acc, te_f1 = run_epoch(test_loader, train=False)
print(f"Test: loss={te_loss:.4f}  accuracy={te_acc:.4f}  F1={te_f1:.4f}")

## 9. Инференс

In [ ]:
@torch.inference_mode()
def predict_batch(texts: list[str]) -> list[tuple[str, float]]:
    model.eval()
    enc = tokenizer(
        texts, max_length=MAX_LEN, padding="max_length",
        truncation=True, return_tensors="pt"
    ).to(DEVICE)
    with autocast(enabled=USE_AMP):
        logits2, _ = model(enc['input_ids'], enc['attention_mask'])
    probs = torch.softmax(logits2, dim=1)[:, 1].cpu()  # P(actionable)
    labels = (probs >= 0.5).long()
    return [(ID2LABEL[l.item()], p.item()) for l, p in zip(labels, probs)]


examples = [
    "Какие вопросы обычно задают на собеседовании?",
    "Расскажи о своём опыте работы с базами данных.",
    "Проект был завершён в срок.",
    "Как работает сборщик мусора в Java?",
    "Я занимался разработкой микросервисов на Go.",
]
for text, (label, conf) in zip(examples, predict_batch(examples)):
    mark = "❓" if label == "actionable" else "💬"
    print(f"{mark} [{conf:.2f}] {text}")

## 10. Сохранение

In [ ]:
torch.save(model.state_dict(), "question_classifier_weights.pth")
with open("model_config.json", "w") as f:
    json.dump({"model_name": MODEL_NAME, "max_len": MAX_LEN, "id2label": ID2LABEL}, f, indent=2)
print("✅ question_classifier_weights.pth + model_config.json")